In [ ]:
import paramiko
import os

In [ ]:
# Directory where the output file will be saved
output_directory = '/home/antonio/UCSC/Research/Server_Mencia_Output'

# Check if the directory exists, if not, create it
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Update the path for the local output file
local_output_path = os.path.join(output_directory, 'output.txt')

In [ ]:
# Server credentials
hostname = 'mencia.soe.ucsc.edu'
username = 'jaguir26'
# Expand the '~' to the full home directory path
private_key_path = os.path.expanduser('~/.ssh/id_rsa')

# File paths
local_script_path = '/home/antonio/UCSC/Research/R_scripts/exDQLM_exe.R'
remote_script_path = 'exDQLM_exe.R'
remote_output_path = 'output.txt'

# Directory where the output file will be saved
output_directory = '/home/antonio/UCSC/Research/Server_Mencia_Output'

# Check if the directory exists, if not, create it
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Update the path for the local output file
local_output_path = os.path.join(output_directory, 'output.txt')

# Create SSH and SFTP clients
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(hostname, username=username, key_filename=private_key_path)
sftp = ssh.open_sftp()

# Upload the R script
sftp.put(local_script_path, remote_script_path)
sftp.close()

# Run the R script using R 4.2.2 and wait for completion
command = f'/opt/R/4.2.2/bin/Rscript {remote_script_path} > {remote_output_path} 2>&1'
stdin, stdout, stderr = ssh.exec_command(command)
exit_status = stdout.channel.recv_exit_status()  # Blocking call
if exit_status == 0:
    print("R script executed successfully")
else:
    print("R script failed with exit status", exit_status)
    
# Download the output
sftp = ssh.open_sftp()
sftp.get(remote_output_path, local_output_path)
sftp.close()

print('Script executed and output file downloaded.')

